## 🤖 Agentic RAG — LLM Agents, Prompt Engineering & Adversarial Attacks

Este notebook implementa un sistema **Agentic RAG** (Retrieval-Augmented Generation) enfocado en **teoría de LLMs**, utilizando **OpenAI GPT** y 3 blog posts de Lilian Weng:
- LLM Powered Autonomous Agents
- Prompt Engineering
- Adversarial Attacks on LLMs

## 0. Setup: instalar dependencias y configurar entorno

In [4]:
%pip install langchain_openai langchain_community langgraph langsmith tiktoken beautifulsoup4 -q

Note: you may need to restart the kernel to use updated packages.


In [5]:
from importlib.metadata import version
print(f"✅ langchain_openai version: {version('langchain-openai')}")
print("✅ Dependencias listas")

✅ langchain_openai version: 1.1.9
✅ Dependencias listas


In [ ]:
import os

# === OpenAI ===
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")  # ← Pon tu API key de OpenAI

# === LangSmith ===
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY", "")
os.environ["LANGCHAIN_PROJECT"] = "01-agentic-rag"

print(f"🔑 OpenAI API Key: {'✅ configurada' if os.environ['OPENAI_API_KEY'] else '❌ no encontrada'}")
print(f"📊 LangSmith Project: {os.environ['LANGCHAIN_PROJECT']}")

## 1. Configurar OpenAI GPT

In [7]:
from langchain_openai import ChatOpenAI

MODEL_ID = "gpt-4.1-mini"

chat_llm = ChatOpenAI(
    model=MODEL_ID,
    temperature=0,
)

# Test rápido
resp = chat_llm.invoke("Say hello in one word.")
print(f"✅ Modelo configurado: {MODEL_ID}")
print(f"Test: {resp.content[:100]}")

✅ Modelo configurado: gpt-4.1-mini
Test: Hello!


## 2. Cargar documentos sobre LLMs y crear vectorstore

Se cargan 3 blog posts de Lilian Weng sobre teoría de LLMs:
- **LLM Powered Autonomous Agents** — agentes, memoria, planificación
- **Prompt Engineering** — técnicas de prompting, chain-of-thought, few-shot
- **Adversarial Attacks on LLMs** — ataques, jailbreaks, defensas

In [8]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

# --- Cargar 3 blog posts de Lilian Weng sobre LLMs ---
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

print("⬇️  Cargando blog posts...")
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]
print(f"📕 {len(docs_list)} documentos cargados")

# --- Split en chunks ---
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=50
)
doc_splits = text_splitter.split_documents(docs_list)
print(f"✂️  {len(doc_splits)} chunks creados")

# --- Embeddings con OpenAI ---
print("\n🔢 Creando embeddings con OpenAI...")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits, embedding=embeddings
)
retriever = vectorstore.as_retriever()
print("✅ Vectorstore creado con documentos sobre LLMs")

USER_AGENT environment variable not set, consider setting it to identify your requests.


⬇️  Cargando blog posts...
📕 3 documentos cargados
✂️  191 chunks creados

🔢 Creando embeddings con OpenAI...
✅ Vectorstore creado con documentos sobre LLMs


## 3. Definir nodos del grafo

El Agentic RAG tiene estos nodos:
1. **Retrieve**: Recupera documentos sobre LLMs del vectorstore
2. **Grade**: Evalúa si los documentos son relevantes (structured output via JSON)
3. **Generate**: Genera respuesta usando el contexto sobre LLMs
4. **Rewrite**: Reformula la query si los docs no son relevantes

In [9]:
from typing import Annotated, Literal, Sequence
from typing_extensions import TypedDict
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph.message import add_messages
import json, re


class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]


# Contador global de rewrites por pregunta (evitar loops infinitos)
_rewrite_count = 0


def retrieve_node(state: AgentState):
    """Recupera documentos del vectorstore."""
    print("---RETRIEVE---")
    question = state["messages"][-1].content
    docs = retriever.invoke(question)
    docs_text = "\n\n".join(doc.page_content for doc in docs)
    return {"messages": [AIMessage(content=docs_text)]}


def grade_documents(state: AgentState) -> Literal["generate", "rewrite"]:
    """Evalúa si los documentos son relevantes a la pregunta."""
    global _rewrite_count
    print("---GRADE DOCUMENTS---")
    messages = state["messages"]
    question = next(m.content for m in messages if isinstance(m, HumanMessage))
    docs = messages[-1].content

    # Si ya hicimos 1 rewrite, aceptar directamente para evitar loops
    if _rewrite_count >= 1:
        print("  ⚡ Max rewrites alcanzado → generate directamente")
        return "generate"

    grading_prompt = f"""You are a grader assessing relevance of a retrieved document to a user question.
Here is the retrieved document (first 1000 chars):
{docs[:1000]}

Here is the user question: {question}

If the document contains keywords or semantic meaning related to the question, it IS relevant.
Respond with ONLY a JSON object: {{"score": "yes"}} or {{"score": "no"}}"""

    try:
        response = chat_llm.invoke(grading_prompt)
        text = response.content
        match = re.search(r'\{[^}]+\}', text)
        if match:
            result = json.loads(match.group())
            score = result.get("score", "yes")
        else:
            score = "yes" if "yes" in text.lower() else "no"
    except:
        score = "yes"

    if score == "yes":
        print("  ✅ DOCS RELEVANT → generate")
        _rewrite_count = 0
        return "generate"
    else:
        print("  ❌ DOCS NOT RELEVANT → rewrite query")
        _rewrite_count += 1
        return "rewrite"


def rewrite(state: AgentState):
    """Reformula la pregunta para mejorar la búsqueda."""
    print("---REWRITE QUERY---")
    question = next(m.content for m in state["messages"] if isinstance(m, HumanMessage))

    prompt = f"""Look at the input and try to reason about the underlying semantic intent / meaning.
Rewrite the following question to be more specific for document retrieval.
Output ONLY the improved question, nothing else.
Original: {question}
Improved:"""

    response = chat_llm.invoke(prompt)
    new_q = response.content.strip().split("\n")[0]
    print(f"  Original:  {question}")
    print(f"  Rewritten: {new_q[:200]}")
    return {"messages": [HumanMessage(content=new_q)]}


def generate(state: AgentState):
    """Genera la respuesta final usando el contexto recuperado."""
    global _rewrite_count
    _rewrite_count = 0  # Reset para la siguiente pregunta
    print("---GENERATE---")
    question = next(m.content for m in state["messages"] if isinstance(m, HumanMessage))
    docs = [m.content for m in state["messages"] if isinstance(m, AIMessage)][-1]

    prompt = f"""You are an assistant for question-answering tasks.
Use the following context to answer the question.
If you don't know the answer, say so. Use three sentences maximum.

Question: {question}

Context:
{docs[:3000]}

Answer:"""

    response = chat_llm.invoke(prompt)
    return {"messages": [AIMessage(content=response.content)]}


print("✅ Nodos definidos: retrieve, grade_documents, rewrite, generate")

✅ Nodos definidos: retrieve, grade_documents, rewrite, generate


## 4. Construir y compilar el grafo

In [10]:
from langgraph.graph import END, StateGraph, START

workflow = StateGraph(AgentState)

# Nodos
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("rewrite", rewrite)
workflow.add_node("generate", generate)

# Edges
workflow.add_edge(START, "retrieve")
workflow.add_conditional_edges(
    "retrieve",
    grade_documents,
    {"generate": "generate", "rewrite": "rewrite"},
)
workflow.add_edge("rewrite", "retrieve")
workflow.add_edge("generate", END)

graph = workflow.compile()
print("✅ Grafo Agentic RAG compilado")

✅ Grafo Agentic RAG compilado


## 5. Ejecutar el agente RAG con preguntas sobre LLMs

In [11]:
questions = [
    "What are the types of agent memory?",
    "What is chain-of-thought prompting?",
    "What are common adversarial attacks on LLMs?",
    "How does task decomposition work in LLM agents?",
    "What is the role of planning in autonomous agents?",
]

for i, question in enumerate(questions, 1):
    print(f"\n{'='*70}")
    print(f"QUESTION {i}: {question}")
    print(f"{'='*70}\n")

    result = graph.invoke(
        {"messages": [HumanMessage(content=question)]},
        config={"recursion_limit": 10}
    )

    # Mostrar documentos recuperados
    ai_msgs = [m for m in result["messages"] if isinstance(m, AIMessage)]
    if len(ai_msgs) >= 2:
        docs_content = ai_msgs[-2].content  # penúltimo AIMessage = docs
        print(f"\n📄 RETRIEVED DOCS (first 300 chars):")
        for j, chunk in enumerate(docs_content.split("\n\n")[:3], 1):
            print(f"  Doc {j}: {chunk[:150]}...")
    
    final = result["messages"][-1]
    answer = final.content if hasattr(final, 'content') else str(final)
    print(f"\n🎯 FINAL ANSWER:\n{answer[:500]}")


QUESTION 1: What are the types of agent memory?

---RETRIEVE---
---GRADE DOCUMENTS---
  ✅ DOCS RELEVANT → generate
---GENERATE---

📄 RETRIEVED DOCS (first 300 chars):
  Doc 1: Memory...
  Doc 2: Short-term memory: I would consider all the in-context learning (See Prompt Engineering) as utilizing short-term memory of the model to learn.
Long-te...
  Doc 3: 
Tool use...

🎯 FINAL ANSWER:
The types of agent memory are sensory memory, short-term memory, and long-term memory. Sensory memory retains raw sensory impressions briefly, short-term memory involves in-context learning within a limited context window, and long-term memory uses an external vector store for extended information retention and fast retrieval.

QUESTION 2: What is chain-of-thought prompting?

---RETRIEVE---
---GRADE DOCUMENTS---
  ✅ DOCS RELEVANT → generate
---GENERATE---

📄 RETRIEVED DOCS (first 300 chars):
  Doc 1: [7] Wei et al. “Chain of thought prompting elicits reasoning in large language models.” NeurIPS 2022
[8] 

## 6. 📊 Evaluación con LangSmith Custom Evaluators

Se crea un **dataset** en LangSmith con las preguntas y se evalúa el RAG con **custom evaluators**:

| Evaluador | Tipo | Qué mide |
|---|---|---|
| `answer_not_empty` | Code | ¿La respuesta no está vacía? |
| `conciseness` | Code | Penaliza respuestas muy largas (1-5) |
| `has_context` | Code | ¿Se usó contexto del retriever? |
| `answer_relevance` | LLM-as-judge | ¿La respuesta es relevante a la pregunta? (GPT) |
| `faithfulness` | LLM-as-judge | ¿La respuesta es fiel al contexto? (GPT) |

In [ ]:
from langsmith import Client, evaluate
from langchain_core.messages import HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from datetime import datetime
import json

client = Client()
PROJECT = "01-agentic-rag"

# ============================================================
# 1. Crear dataset en LangSmith con preguntas + respuestas esperadas
# ============================================================
DATASET_NAME = "agentic-rag-llm-questions"

# Eliminar dataset anterior si existe
try:
    existing = client.read_dataset(dataset_name=DATASET_NAME)
    client.delete_dataset(dataset_id=existing.id)
    print(f"🗑️  Dataset anterior '{DATASET_NAME}' eliminado")
except:
    pass

examples = [
    {
        "inputs": {"question": "What are the types of agent memory?"},
        "outputs": {"expected_topics": "sensory, short-term, long-term, working memory"},
    },
    {
        "inputs": {"question": "What is chain-of-thought prompting?"},
        "outputs": {"expected_topics": "step by step reasoning, intermediate steps, few-shot"},
    },
    {
        "inputs": {"question": "What are common adversarial attacks on LLMs?"},
        "outputs": {"expected_topics": "jailbreak, prompt injection, data poisoning"},
    },
    {
        "inputs": {"question": "How does task decomposition work in LLM agents?"},
        "outputs": {"expected_topics": "subgoals, chain of thought, decompose complex tasks"},
    },
    {
        "inputs": {"question": "What is the role of planning in autonomous agents?"},
        "outputs": {"expected_topics": "subgoal decomposition, reflection, refinement"},
    },
]

dataset = client.create_dataset(dataset_name=DATASET_NAME)
client.create_examples(dataset_id=dataset.id, examples=examples)
print(f"✅ Dataset creado: '{DATASET_NAME}' con {len(examples)} ejemplos")

# ============================================================
# 2. Target function — ejecuta el Agentic RAG
# ============================================================
def rag_target(inputs: dict) -> dict:
    """Ejecuta el grafo RAG y retorna la respuesta + contexto."""
    question = inputs["question"]
    result = graph.invoke(
        {"messages": [HumanMessage(content=question)]},
        config={"recursion_limit": 10}
    )

    ai_msgs = [m for m in result["messages"] if isinstance(m, AIMessage)]
    answer = ai_msgs[-1].content if ai_msgs else ""
    context = ai_msgs[-2].content if len(ai_msgs) >= 2 else ""

    return {
        "answer": answer,
        "context": context[:2000],
    }

# ============================================================
# 3. Custom Evaluators
# ============================================================

# --- Evaluador 1: respuesta no vacía ---
def answer_not_empty(outputs: dict) -> bool:
    """Verifica que la respuesta no esté vacía."""
    answer = outputs.get("answer", "")
    return len(str(answer).strip()) > 10

# --- Evaluador 2: concisión (1-5, menor = mejor) ---
def conciseness(outputs: dict) -> dict:
    """Evalúa concisión: 1=muy conciso, 5=demasiado largo."""
    answer = outputs.get("answer", "")
    length = len(str(answer))
    if length < 200:
        score = 1
    elif length < 500:
        score = 2
    elif length < 1000:
        score = 3
    elif length < 2000:
        score = 4
    else:
        score = 5
    return {"key": "conciseness", "score": score}

# --- Evaluador 3: ¿tiene contexto? ---
def has_context(outputs: dict) -> bool:
    """Verifica que se haya recuperado contexto del retriever."""
    context = outputs.get("context", "")
    return len(str(context).strip()) > 50

# --- Evaluador 4: relevancia (LLM-as-judge con GPT) ---
judge_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

def answer_relevance(inputs: dict, outputs: dict) -> dict:
    """Usa GPT para evaluar si la respuesta es relevante a la pregunta."""
    prompt = f"""You are an evaluation judge. Rate how relevant the answer is to the question.

Question: {inputs['question']}

Answer: {str(outputs.get('answer', ''))[:1000]}

Rate relevance on a scale 0-1:
- 1.0 = directly and fully answers the question
- 0.5 = partially relevant
- 0.0 = completely irrelevant

Respond with ONLY a JSON object: {{"score": <float>}}"""

    response = judge_llm.invoke(prompt)
    try:
        import re
        match = re.search(r'\{[^}]+\}', response.content)
        if match:
            result = json.loads(match.group())
            return {"key": "answer_relevance", "score": float(result.get("score", 0.5))}
    except:
        pass
    return {"key": "answer_relevance", "score": 0.5}

# --- Evaluador 5: fidelidad al contexto (LLM-as-judge) ---
def faithfulness(inputs: dict, outputs: dict) -> dict:
    """Usa GPT para evaluar si la respuesta es fiel al contexto recuperado."""
    context = str(outputs.get("context", ""))[:1500]
    answer = str(outputs.get("answer", ""))[:1000]

    prompt = f"""You are an evaluation judge. Rate how faithful the answer is to the provided context.
A faithful answer only contains information that can be derived from the context.

Question: {inputs['question']}

Context (retrieved documents):
{context}

Answer: {answer}

Rate faithfulness on a scale 0-1:
- 1.0 = fully grounded in the context
- 0.5 = partially grounded, some info not in context
- 0.0 = answer contradicts or ignores the context

Respond with ONLY a JSON object: {{"score": <float>}}"""

    response = judge_llm.invoke(prompt)
    try:
        import re
        match = re.search(r'\{[^}]+\}', response.content)
        if match:
            result = json.loads(match.group())
            return {"key": "faithfulness", "score": float(result.get("score", 0.5))}
    except:
        pass
    return {"key": "faithfulness", "score": 0.5}

# ============================================================
# 4. Ejecutar evaluación
# ============================================================
print("\n🚀 Ejecutando evaluación con LangSmith...")
print(f"   Dataset: {DATASET_NAME}")
print(f"   Evaluadores: answer_not_empty, conciseness, has_context, answer_relevance, faithfulness")
print(f"   Judge LLM: gpt-4.1-mini")
print()

evaluator_list = [  # type: ignore[list-item]
    answer_not_empty,
    conciseness,
    has_context,
    answer_relevance,
    faithfulness,
]

results = evaluate(
    rag_target,
    data=DATASET_NAME,
    evaluators=evaluator_list,
    experiment_prefix="agentic-rag-eval",
    description="Evaluación del Agentic RAG con custom evaluators + LLM-as-judge",
    max_concurrency=1,  # secuencial para evitar rate limits
)

# ============================================================
# 5. Mostrar resultados
# ============================================================
print(f"\n{'='*85}")
print(f"📊 RESULTADOS DE EVALUACIÓN — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*85}")
print(f"\n🌐 Ver resultados detallados en: https://smith.langchain.com")
print(f"📁 Dataset: {DATASET_NAME}")
print(f"📁 Proyecto: {PROJECT}")
print(f"\n✅ Evaluación completada. Revisa los scores en LangSmith UI.")

✅ Dataset creado: 'agentic-rag-llm-questions' con 5 ejemplos

🚀 Ejecutando evaluación con LangSmith...
   Dataset: agentic-rag-llm-questions
   Evaluadores: answer_not_empty, conciseness, has_context, answer_relevance, faithfulness
   Judge LLM: gpt-4.1-mini



/Users/santiagomartinezdie/Documents/research/rag_example/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'agentic-rag-eval-70fb2bdc' at:
https://smith.langchain.com/o/683e946d-2c38-4fb8-9301-35f284d74e49/datasets/03a1b083-c93a-4de7-80b8-a5090cda48e1/compare?selectedSessions=77d26705-3e51-4256-a32c-7adc58b0a437




0it [00:00, ?it/s]

---RETRIEVE---
---GRADE DOCUMENTS---
  ✅ DOCS RELEVANT → generate
---GENERATE---
---RETRIEVE---
---GRADE DOCUMENTS---
  ✅ DOCS RELEVANT → generate
---GENERATE---


...

---RETRIEVE---
---GRADE DOCUMENTS---
  ✅ DOCS RELEVANT → generate
---GENERATE---


3it [00:09,  2.88s/it]

---RETRIEVE---
---GRADE DOCUMENTS---
  ✅ DOCS RELEVANT → generate
---GENERATE---


5it [00:13,  2.77s/it]


📊 RESULTADOS DE EVALUACIÓN — 2026-02-15 19:59:58

🌐 Ver resultados detallados en: https://smith.langchain.com
📁 Dataset: agentic-rag-llm-questions
📁 Proyecto: 01-agentic-rag

✅ Evaluación completada. Revisa los scores en LangSmith UI.


## 7. Evaluación universal del grafo RAG con LangSmith (evaluador desacoplado)

Esta celda permite evaluar el grafo RAG con los evaluadores estándar de LangSmith, independientemente de la arquitectura.

In [13]:
# 1. Importa el evaluador universal
from rag_evaluator import evaluate_rag

# 2. Wrapper para tu grafo
def graph_rag_fn(inputs):
    result = graph.invoke(
        {"messages": [HumanMessage(content=inputs["question"])]},
        config={"recursion_limit": 10}
    )
    ai_msgs = [m for m in result["messages"] if isinstance(m, AIMessage)]
    answer = ai_msgs[-1].content if ai_msgs else ""
    docs = ai_msgs[-2].content if len(ai_msgs) >= 2 else ""
    return {"answer": answer, "documents": [docs]}

# 3. Dataset de ejemplo (puedes ampliarlo)
dataset = [
    {"inputs": {"question": "What are the types of agent memory?"}, "outputs": {"answer": "sensory, short-term, long-term, working memory"}},
    {"inputs": {"question": "What is chain-of-thought prompting?"}, "outputs": {"answer": "step by step reasoning, intermediate steps, few-shot"}},
    {"inputs": {"question": "What are common adversarial attacks on LLMs?"}, "outputs": {"answer": "jailbreak, prompt injection, data poisoning"}},
    {"inputs": {"question": "How does task decomposition work in LLM agents?"}, "outputs": {"answer": "subgoals, chain of thought, decompose complex tasks"}},
    {"inputs": {"question": "What is the role of planning in autonomous agents?"}, "outputs": {"answer": "subgoal decomposition, reflection, refinement"}},
]

# 4. Ejecuta la evaluación
evaluate_rag(graph_rag_fn, dataset, dataset_name="agentic-rag-eval", project="agentic-rag")

✅ Dataset creado: 'agentic-rag-eval' con 5 ejemplos

🚀 Ejecutando evaluación con evaluadores: ['correctness', 'relevance', 'groundedness', 'retrieval_relevance']
View the evaluation results for experiment: 'agentic-rag-6f42c4b9' at:
https://smith.langchain.com/o/683e946d-2c38-4fb8-9301-35f284d74e49/datasets/ff8a1d75-2685-4971-83d2-1c2f8fe875ec/compare?selectedSessions=c8beeebe-16a7-47fc-93a1-fae683ef81d4




0it [00:00, ?it/s]

---RETRIEVE---
---GRADE DOCUMENTS---
  ✅ DOCS RELEVANT → generate
---GENERATE---


1it [00:15, 15.34s/it]

---RETRIEVE---
---GRADE DOCUMENTS---
  ✅ DOCS RELEVANT → generate
---GENERATE---


...

4it [00:53, 13.19s/it]

---RETRIEVE---
---GRADE DOCUMENTS---
  ✅ DOCS RELEVANT → generate
---GENERATE---


5it [01:07, 13.57s/it]


🌐 Ver resultados detallados en: https://smith.langchain.com
📁 Dataset: agentic-rag-eval
📁 Proyecto: agentic-rag


<ExperimentResults agentic-rag-6f42c4b9>